# Multi-Cycle Time-Dependent Fitting

This notebook shows how to fit repeating, multi-phase dynamics by passing a **list of models** to `add_time_dependence`. The example system is the same as in [`01_basic_fitting`](../01_basic_fitting/example.ipynb): a `GLP` peak on a linear background.<br>
However, after t = 0 the peak position now alternates between two repeating subcycles: a negative shift that relaxes quickly and a mirrored positive shift that relaxes more slowly, all blurred by a Gaussian instrument response function (IRF).

The two rules that drive everything here:

1. **The `dynamics_model` list.** Element 0 applies *globally* (all time points) and is typically used for an IRF convolution. All following elements alternate as subcycles within each repetition period. If `N` is the length of the `dynamics_model` list, then `N-1` is the number of subcycles.
2. **The `frequency` parameter.** Simply 1/T of the full repeating period. The `N−1` subcycle models split each period equally, so one subcycle lasts `1/(frequency*(N-1))`. Subcycles start at t = 0; everything before is baseline.

Workflow:
1. **Select the data**: load, inspect, set fit limits.
2. **Fit the baseline** (`base`): ground-state spectrum from pre-trigger slices.
3. **Slice-by-slice fitting** (`SbS`): reveal the alternating sawtooth that motivates the multi-cycle model.
4. **Multi-cycle 2D fit** (`2D`): one global fit with `['IRF', 'MonoExpNeg', 'MonoExpPos']` repeating at `frequency=0.25`.
5. **Inspect the results**: compare to the known truth (`data/models_energy_truth.yaml`, `data/models_time_truth.yaml`).

In [ ]:
from pathlib import Path

import numpy as np
import trspecfit

## 1. Load and Inspect Data

### 1.1 Load Data

Load simulated time- and energy-resolved spectroscopy data from CSV files. The data is synthetic, generated by [`data/generate_data.ipynb`](data/generate_data.ipynb) which can be re-run to regenerate or change the data.
<br><br>
The 2D dataset is a stack of 1D energy spectra, one per time delay. The `data` array is `[time, energy]`, i.e. rows are time, columns are energy.

In [ ]:
# Create project
project = trspecfit.Project(path=Path.cwd())

# Directory containing data files
data_folder = 'data'

# Add file to project
file = trspecfit.File(
    parent_project=project,
    path=data_folder,
    data=np.loadtxt(project.path / data_folder / 'data.csv', delimiter=','),
    energy=np.loadtxt(project.path / data_folder / 'energy.csv'),
    time=np.loadtxt(project.path / data_folder / 'time.csv')
)

### 1.2 Inspect Data

A `GLP` peak sits on a linear background (`LinBack`). After t = 0 the peak position alternates periodically: it jumps down and relaxes back (fast), then jumps up by the same amount and relaxes back (slower), repeating every 4 time units. The sharp jumps are rounded by the Gaussian IRF.

In [ ]:
# Visualize full dataset
file.describe()

### 1.3 Define Fitting Region

Set energy and time limits for the fits. They appear as black, dotted lines in the plot.

In [ ]:
# Set fitting limits (absolute values)
file.set_fit_limits(
    energy_limits=[4, 14],  # Energy range
    time_limits=[-4, 15.75]  # Time range
)

## 2. Fit Baseline / Ground State Spectrum

### 2.1 Define Baseline Spectrum

Extract the ground-state spectrum by averaging pre-trigger time slices. The Gaussian IRF is non-causal — it bleeds the first subcycle's onset slightly *before* t = 0 — so stop the baseline well before t = 0 (here at −0.5, about 3 IRF widths).

In [ ]:
# Define baseline using absolute time values
file.define_baseline(
    time_start=-4,
    time_stop=-0.5,
    time_type='abs'  # Use absolute time values (or 'ind' for indices)
)

### 2.2 Fit Baseline Spectrum

Fit the ground state spectrum leaving most fit parameters free to vary. The model is defined in `models_energy.yaml`.
<br><br>
During time-dependent fits, the values of all fixed parameters (`vary=False`) will be set to the baseline fit result values.

In [ ]:
# Load baseline model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='base'
)

# Inspect model structure and parameters
file.describe_model(model_info='base', detail=0)

In [ ]:
# Fit baseline model to data
file.fit_baseline(
    model_name='base',
    stages=2,
    try_ci=0  # Confidence intervals: see 12_uncertainty_mcmc
)

## 3. Slice-by-Slice Fitting

Fit each time slice independently with the peak position `GLP_01_x0` as the only free parameter. The result is the position-vs-time trace *without* assuming any functional form — look for the **alternating sawtooth**: downward jumps at t = 0, 4, 8, … and upward jumps at t = 2, 6, 10, …, each relaxing back toward the baseline. That repeating two-phase pattern is what motivates a multi-cycle model; the different relaxation speeds of the two phases motivate per-subcycle time constants.

In [ ]:
# Load Slice-by-Slice model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='SbS'
)

file.describe_model('SbS', detail=0)

In [ ]:
# Perform Slice-by-Slice fit
file.fit_slice_by_slice(
    model_name='SbS',
    stages=1,
    try_ci=0  # Minimizer needs >=2 varying parameters to determine confidence intervals
)

## 4. Multi-Cycle Global 2D Fitting

Instead of one time law for the whole dataset (as in `01_basic_fitting`), pass a **list** of models from `models_time.yaml`:

```python
dynamics_model=['IRF', 'MonoExpNeg', 'MonoExpPos'], frequency=0.25
```

- **`IRF`** (element 0) applies to **all** time points: a `gaussCONV` that convolves the full assembled time trace. A model whose only component is a convolution is valid exactly for this slot.
- **`MonoExpNeg`** / **`MonoExpPos`** (elements 1+) alternate as subcycles. `frequency=0.25` sets the full period to 1/0.25 = 4 time units, which the two subcycle models split equally: 2 units each, starting at t = 0.
- **Each subcycle runs on a local clock.** Inside a subcycle model, t = 0 is the *start of that subcycle*, in every repetition. That is why `t0: [0, False]` is correct here — tau is measured from each subcycle's own start, not from the global trigger.

**Linked parameters across subcycles** (also see [`02_dependent_parameters`](../02_dependent_parameters/example.ipynb)): `MonoExpPos` mirrors the amplitude via the expression `A: ["-expFun_01_A"]` — one shared amplitude, no extra free parameter — while its `tau` stays free, so each subcycle gets its own relaxation time. Expressions work across subcycle models exactly like across energy components.

In [ ]:
# Load 2D model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='2D'
)

file.describe_model(model_info='2D', detail=0)

In [ ]:
# Add multi-cycle time dependence to the peak position
file.add_time_dependence(
    target_model='2D',
    target_parameter='GLP_01_x0',
    dynamics_yaml='models_time.yaml',
    dynamics_model=[
        'IRF',         # element 0: global IRF convolution (all time points)
        'MonoExpNeg',  # subcycle 1: fast negative shift
        'MonoExpPos'   # subcycle 2: mirrored amplitude (expression), own tau
    ],
    frequency=0.25,  # full period T = 1/0.25 = 4; two subcycles of 2 each
)

print("\n=== Model with Multi-Cycle Time Dependence ===")
file.describe_model(model_info='2D', detail=1)

In [ ]:
# Perform global 2D fit
file.fit_2d(
    model_name='2D',
    stages=2,
    try_ci=0  # Skip confidence interval estimation for a faster fit
)

## 5. Inspect the Fit Results

`file.get_fit_results(fit_type='2d')` returns a `pandas.DataFrame` with one row per parameter. The subcycle parameters are named by their *component*, numbered in list order across the dynamics models — `GLP_01_x0_expFun_01_*` is the `expFun` of `MonoExpNeg` (subcycle 1), `GLP_01_x0_expFun_02_*` the one of `MonoExpPos` (subcycle 2); the YAML model names themselves do not appear in parameter names. Note `expFun_02_A` keeps its expression in the `expr` column — the mirrored amplitude was never fit independently — while the two `tau` values are separate free parameters.

Because the data is synthetic, the results can be compared to the known truth (`data/models_*_truth.yaml`): the global fit recovers the IRF width (`SD ≈ 0.15`), the shared shift amplitude (`A ≈ −2`), and the two distinct relaxation times (`tau ≈ 0.4` in subcycle 1 vs `≈ 0.8` in subcycle 2).
<br><br>
One subtlety worth noticing in the fitted curves: with the same amplitude, the *visible* excursion of subcycle 1 is smaller than subcycle 2's — its faster relaxation (tau comparable to the IRF width) erodes more of the jump under the convolution. The fit still recovers the true shared amplitude because the IRF is part of the model.

In [ ]:
file.get_fit_results(fit_type='2d')

## Tips for Multi-Cycle Fitting

**Identifying subcycles**
- Use Slice-by-Slice fits to reveal the repeating pattern and count the distinct phases per period.
- Set `frequency` from the experiment (pulse timing, modulation frequency) whenever it is known.

**Structuring the `dynamics_model` list**
- Element 0 is global. Put the IRF there (`gaussCONV`-only model), or a model containing only `none: {}` if there is no global effect.
- Elements 1+ split each 1/`frequency` period equally and run on local clocks that reset at each subcycle start.
- Link physically shared parameters across subcycle models with expressions (e.g. `"-expFun_01_A"`); keep genuinely independent ones (here: the two `tau`) free.

**Strategy**
- Fit the baseline first; pin line-shape parameters during the 2D fit.
- Start with the simple per-subcycle models and add complexity if the residuals demand it.

## Next Steps

- Which parameters are truly independent across subcycles? The drive scheme tells you the subcycle structure, but not whether the responses share parameters — fit a fully-linked variant (e.g. shared `tau`) against the independent one and compare: [`10_model_comparison`](../10_model_comparison/example.ipynb)
- Parameter expressions in depth: [`02_dependent_parameters`](../02_dependent_parameters/example.ipynb)
- Parameters that vary along an auxiliary axis: [`04_parameter_profiles`](../04_parameter_profiles/example.ipynb)
- Uncertainty estimation (MCMC): [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb)
- Save, load, and export fits: [`11_save_load_export`](../11_save_load_export/example.ipynb)